Expected format

```py
{
    "article":  "Some text..."
    "summary":  "Summary of some text..."
}
```

## 0. Load imports

In [ ]:
import os
import numpy as np
import torch
import evaluate

from datasets import DatasetDict, load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42
set_seed(SEED)

## 1. Loading the data

In [ ]:

from pathlib import Path

_NB_CWD = Path.cwd()
REPO_ROOT = _NB_CWD.parent if _NB_CWD.name == "notebooks" else _NB_CWD
ROOT = str(REPO_ROOT)

DATA = os.path.join(ROOT, "data")
PROCESSED = os.path.join(DATA, "processed")
SUMMARY = os.path.join(PROCESSED, "summary")

DATASET_NAME = "NS"
NS = os.path.join(SUMMARY, DATASET_NAME)

_raw = load_dataset(
    "csv",
    data_files={"train": os.path.join(NS, "full.csv")},
)["train"]

_cols = set(_raw.column_names)
if {"ctext", "text"}.issubset(_cols):
    _src_article, _src_summary = "ctext", "text"
elif {"article", "summary"}.issubset(_cols):
    _src_article, _src_summary = "article", "summary"
else:
    raise ValueError(f"Unexpected NS columns: {_raw.column_names}")

_raw = _raw.filter(
    lambda ex, a=_src_article, s=_src_summary: (
        ex.get(a) is not None
        and ex.get(s) is not None
        and str(ex[a]).strip() != ""
        and str(ex[s]).strip() != ""
    )
)
if (_src_article, _src_summary) != ("article", "summary"):
    _raw = _raw.rename_columns(
        {_src_article: "article", _src_summary: "summary"}
    )
_drop = [c for c in _raw.column_names if c not in ("article", "summary")]
if _drop:
    _raw = _raw.remove_columns(_drop)

# NS ships as one flat CSV; do a deterministic 80/10/10 split.
_first = _raw.train_test_split(test_size=0.2, seed=SEED)
_second = _first["test"].train_test_split(test_size=0.5, seed=SEED)
dataset = DatasetDict(
    {
        "train": _first["train"],
        "validation": _second["train"],
        "test": _second["test"],
    }
)


for _split in ("train", "validation", "test"):
    dataset[_split] = dataset[_split].shuffle(seed=SEED)

print(
    f"{DATASET_NAME}: train={len(dataset['train'])}, "
    f"val={len(dataset['validation'])}, test={len(dataset['test'])}"
)

## 2. Fine-tuning a Model

In [ ]:
base_model = "facebook/bart-base"
hf_token = os.getenv("HF_TOKEN")
hf_kwargs = {"token": hf_token} if hf_token else {}

tokenizer = BartTokenizer.from_pretrained(base_model, **hf_kwargs)
model = BartForConditionalGeneration.from_pretrained(base_model, **hf_kwargs)

model.generation_config.max_length = 128
model.generation_config.min_length = 55
model.generation_config.num_beams = 4
model.generation_config.length_penalty = 2.0
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.early_stopping = True

In [ ]:

max_input_length = 768
max_target_length = 128


def tokenize(example):
    model_inputs = tokenizer(
        example["article"],
        max_length=max_input_length,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        example["summary"],
        max_length=max_target_length,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["article", "summary"],
)



In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

In [ ]:
learning_rate = 3e-5
num_epochs = 10
weight_decay = 0.01
warmup_ratio = 0.1

use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

train_bs = 32 if use_cuda else 1
eval_bs = 8 if use_cuda else 1
grad_accum = 2 if use_cuda else 1
num_workers = min(8, os.cpu_count() or 1)

# Keep all training artefacts under <repo>/results so notebooks/ stays clean.
RESULTS_DIR = os.path.join(ROOT, "results")
CKPT_DIR = os.path.join(
    RESULTS_DIR, "checkpoints", f"bart-finetuned-{DATASET_NAME.lower()}"
)
os.makedirs(CKPT_DIR, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    learning_rate=learning_rate,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=max_target_length,
    generation_num_beams=4,
    logging_steps=10,
    max_grad_norm=0.5,
    dataloader_num_workers=num_workers,
    dataloader_pin_memory=use_cuda,
    report_to="none",
    bf16=use_cuda,
    fp16=False,
    tf32=use_cuda,
    disable_tqdm=True,
)

## 3. Evaluation

In [ ]:
rouge = evaluate.load("rouge")


def metrics(eval_pred):
    pred, labels = eval_pred

    if isinstance(pred, tuple):
        pred = pred[0]

    pred = np.asarray(pred)
    labels = np.asarray(labels)

    if pred.ndim == 3:
        pred = np.argmax(pred, axis=-1)

    pad_id = tokenizer.pad_token_id
    vocab_size = tokenizer.vocab_size

    pred = pred.astype(np.int64)
    pred = np.where(pred < 0, pad_id, pred)
    pred = np.where(pred >= vocab_size, pad_id, pred)

    labels = np.where(labels != -100, labels, pad_id)
    labels = labels.astype(np.int64)

    decoded_pred = tokenizer.batch_decode(
        pred.tolist(),
        skip_special_tokens=True,
    )
    decoded_labels = tokenizer.batch_decode(
        labels.tolist(),
        skip_special_tokens=True,
    )

    decoded_pred = [x.strip() for x in decoded_pred]
    decoded_labels = [x.strip() for x in decoded_labels]

    scores = rouge.compute(
        predictions=decoded_pred,
        references=decoded_labels,
        use_stemmer=True,
    )

    return {
        "rouge1": round(scores["rouge1"], 3),
        "rouge2": round(scores["rouge2"], 3),
        "rougeL": round(scores["rougeL"], 3),
        "rougeLsum": round(scores["rougeLsum"], 3),
    }

## 4. Training

In [ ]:
import json
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

last_ckpt = (
    get_last_checkpoint(args.output_dir)
    if os.path.isdir(args.output_dir)
    else None
)
if last_ckpt:
    print(f"Resuming from {last_ckpt}")
else:
    print("Starting fresh (no checkpoint found)")

train_output = trainer.train(resume_from_checkpoint=last_ckpt)
print("train output:", train_output)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval_rougeL during training: {trainer.state.best_metric}")

gen_kwargs = dict(
    max_length=max_target_length,
    min_length=55,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
)

eval_metrics = trainer.evaluate(
    eval_dataset=tokenized["validation"],
    metric_key_prefix="final_val",
    **gen_kwargs,
)
print(f"[{DATASET_NAME}] Validation (best model):", eval_metrics)

test_metrics = trainer.evaluate(
    eval_dataset=tokenized["test"],
    metric_key_prefix="final_test",
    **gen_kwargs,
)
print(f"[{DATASET_NAME}] Test (best model):", test_metrics)

final_dir = os.path.join(RESULTS_DIR, f"bart-final-{DATASET_NAME.lower()}")
trainer.save_model(final_dir)
print(f"Final model saved to {final_dir}")

REPORT_DIR = os.path.join(RESULTS_DIR, "report")
os.makedirs(REPORT_DIR, exist_ok=True)
final_metrics_path = os.path.join(
    REPORT_DIR, f"final_metrics_{DATASET_NAME.lower()}.json"
)
with open(final_metrics_path, "w") as f:
    json.dump(
        {
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "best_metric_eval_rougeL": trainer.state.best_metric,
            "validation": eval_metrics,
            "test": test_metrics,
        },
        f,
        indent=2,
        default=str,
    )
print(f"Final metrics saved to {final_metrics_path}")

## 5. Inference

In [ ]:
article = (
    "A drunk teenage boy had to be rescued by security after jumping into a lions' enclosure "
    "at a zoo in western India. Rahul Kumar, 17, clambered over the enclosure fence at the "
    "Kamla Nehru Zoological Park in Ahmedabad, and began running towards the animals, shouting "
    "he would 'kill them'. Mr Kumar explained afterwards that he was drunk and 'thought I'd "
    "stand a good chance' against the predators. Next level drunk: Intoxicated Rahul Kumar, 17, "
    "climbed into the lions' enclosure at a zoo in Ahmedabad and began running towards the animals "
    "shouting 'Today I kill a lion!' Mr Kumar had been sitting near the enclosure when he suddenly "
    "made a dash for the lions, surprising zoo security. The intoxicated teenager ran towards the lions, "
    "shouting: 'Today I kill a lion or a lion kills me!' A zoo spokesman said: 'Guards had earlier spotted "
    "him close to the enclosure but had no idea he was planing to enter it. Fortunately, there are eight "
    "moats to cross before getting to where the lions usually are and he fell into the second one, allowing "
    "guards to catch up with him and take him out. We then handed him over to the police.' Brave fool: "
    "Fortunately, Mr Kumar fell into a moat as he ran towards the lions and could be rescued by zoo security "
    "staff before reaching the animals. Kumar later explained: 'I don't really know why I did it. I was drunk "
    "and thought I'd stand a good chance.' A police spokesman said: 'He has been cautioned and will be sent "
    "for psychiatric evaluation. Fortunately for him, the lions were asleep and the zoo guards acted quickly "
    "enough to prevent a tragedy similar to that in Delhi.' Last year a 20-year-old man was mauled to death "
    "by a tiger in the Indian capital after climbing into its enclosure at the city zoo."
)

reference_summary = (
    "Drunk teenage boy climbed into lion enclosure at zoo in west India. "
    "Rahul Kumar, 17, ran towards animals shouting 'Today I kill a lion!' "
    "Fortunately he fell into a moat before reaching lions and was rescued."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

inputs = tokenizer(
    article,
    return_tensors="pt",
    max_length=max_input_length,
    truncation=True,
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=max_target_length,
    min_length=55,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True,
)

print(generated_summary)

## 6. Final Evaluation

In [ ]:
print(f"Training on {DATASET_NAME} finished.")
print(f"\nBest checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval_rougeL during training: {trainer.state.best_metric}")
print("\nValidation metrics (best model on full val):")
print(eval_metrics)
print("\nTest metrics (best model on full test):")
print(test_metrics)

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

rouge_keys = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
log_history = trainer.state.log_history

train_steps, train_losses = [], []
for e in log_history:
    if "loss" in e and "eval_loss" not in e and "epoch" in e:
        train_steps.append(e["step"])
        train_losses.append(e["loss"])

per_epoch_train = defaultdict(list)
for e in log_history:
    if "loss" in e and "eval_loss" not in e and "epoch" in e:
        per_epoch_train[int(np.ceil(e["epoch"]))].append(e["loss"])
train_loss_epoch = {ep: float(np.mean(v)) for ep, v in per_epoch_train.items()}

eval_loss_epoch, eval_rougeL_epoch = {}, {}
for e in log_history:
    if "eval_loss" in e and "epoch" in e:
        ep = int(round(e["epoch"]))
        eval_loss_epoch[ep] = e["eval_loss"]
        if "eval_rougeL" in e:
            eval_rougeL_epoch[ep] = e["eval_rougeL"]

epochs_train = sorted(train_loss_epoch)
epochs_eval = sorted(eval_loss_epoch)
epochs_rouge = sorted(eval_rougeL_epoch)

window = max(1, len(train_losses) // 100)
smoothed = [
    sum(train_losses[max(0, i - window): i + 1]) / (i - max(0, i - window) + 1)
    for i in range(len(train_losses))
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(train_steps, train_losses, alpha=0.25, label="raw")
ax.plot(train_steps, smoothed, color="C0", label=f"moving avg (w={window})")
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title(f"{DATASET_NAME}: training loss (per step)")
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[0, 1]
val_scores = [eval_metrics[f"final_val_{k}"] for k in rouge_keys]
test_scores = [test_metrics[f"final_test_{k}"] for k in rouge_keys]
splits = [("validation", val_scores, "C0"), ("test", test_scores, "C1")]
x = np.arange(len(rouge_keys))
width = 0.8 / len(splits)
for i, (name, scores, color) in enumerate(splits):
    offset = (i - (len(splits) - 1) / 2) * width
    bars = ax.bar(x + offset, scores, width, label=name, color=color)
    for bar, s in zip(bars, scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{s:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
ax.set_xticks(x)
ax.set_xticklabels(rouge_keys)
ax.set_ylabel("score")
ax.set_title(f"{DATASET_NAME}: best model ROUGE")
ax.set_ylim(0, max(max(val_scores), max(test_scores)) * 1.15)
ax.grid(True, axis="y", alpha=0.3)
ax.legend()

ax = axes[1, 0]
ax.plot(epochs_train, [train_loss_epoch[e] for e in epochs_train], marker="o", color="C0", label="train_loss")
ax.plot(epochs_eval, [eval_loss_epoch[e] for e in epochs_eval], marker="s", color="C1", label="eval_loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title(f"{DATASET_NAME}: loss per epoch")
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1, 1]
ax.plot(epochs_train, [train_loss_epoch[e] for e in epochs_train], marker="o", color="C0", label="train_loss")
ax.plot(epochs_eval, [eval_loss_epoch[e] for e in epochs_eval], marker="s", color="C1", label="eval_loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.grid(True, alpha=0.3)
ax2 = ax.twinx()
ax2.plot(epochs_rouge, [eval_rougeL_epoch[e] for e in epochs_rouge], marker="^", color="C2", label="eval_rougeL")
ax2.set_ylabel("eval_rougeL")
ax2.set_ylim(0, 1)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="best")
ax.set_title(f"{DATASET_NAME}: loss & ROUGE-L per epoch")

plt.tight_layout()
fig_path = f"training_results_{DATASET_NAME.lower()}.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to {fig_path}")

## 8. Tests

Sanity tests for the fine-tuned model. They run against the in-memory
`model` / `tokenizer` / `metrics` and a small slice of the validation set,
so they stay fast (a few seconds) and don't need any extra fixtures.

In [ ]:
import math
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)


def _summarise(
    text,
    min_length=55,
    max_length=None,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
):
    enc = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True,
    ).to(device)
    out = model.generate(
        enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_length=max_length or max_target_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=length_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        early_stopping=True,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


# ---------- tests ----------

def test_tokenizer_roundtrip():
    text = "The quick brown fox jumps over the lazy dog."
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    decoded = tokenizer.decode(ids, skip_special_tokens=True).strip()
    assert decoded.lower() == text.lower(), f"got: {decoded!r}"


def test_model_forward_shapes_and_finite_loss():
    sample = dataset["validation"][0]
    enc = tokenizer(
        sample["article"],
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True,
    ).to(device)
    lab = tokenizer(
        sample["summary"],
        return_tensors="pt",
        max_length=max_target_length,
        truncation=True,
    )["input_ids"].to(device)

    with torch.no_grad():
        out = model(**enc, labels=lab)

    assert out.logits.shape[0] == 1, out.logits.shape
    assert out.logits.shape[-1] == tokenizer.vocab_size, out.logits.shape
    assert math.isfinite(out.loss.item()), f"loss={out.loss.item()}"


def test_generation_non_empty_and_within_length():
    sample = dataset["validation"][0]
    summary = _summarise(sample["article"])
    assert summary, "generated summary is empty"
    n_tokens = len(tokenizer(summary, add_special_tokens=False)["input_ids"])
    assert n_tokens <= max_target_length, n_tokens
    assert n_tokens >= 8, f"summary suspiciously short: {n_tokens} tokens"


def test_generation_compresses_article():
    sample = dataset["validation"][0]
    article_tokens = len(
        tokenizer(sample["article"], add_special_tokens=False)["input_ids"]
    )
    summary = _summarise(sample["article"])
    summary_tokens = len(
        tokenizer(summary, add_special_tokens=False)["input_ids"]
    )
    ratio = summary_tokens / max(1, article_tokens)
    assert ratio < 0.85, (
        f"summary not meaningfully shorter than article (ratio={ratio:.2f})"
    )


def test_different_articles_yield_different_summaries():
    samples = [dataset["validation"][i]["article"] for i in range(3)]
    summaries = [_summarise(s) for s in samples]
    assert len(set(summaries)) == len(summaries), (
        "model collapsed to identical outputs:\n" + "\n---\n".join(summaries)
    )


def test_generation_is_deterministic_with_beam_search():
    article = dataset["validation"][0]["article"]
    a = _summarise(article)
    b = _summarise(article)
    assert a == b, f"non-deterministic outputs:\n  a={a!r}\n  b={b!r}"


def test_metrics_perfect_score_on_identical_inputs():
    refs = [
        "the cat sat on the mat",
        "summarisation models compress text",
    ]
    pred_ids = [
        tokenizer(s, add_special_tokens=False)["input_ids"] for s in refs
    ]
    label_ids = [list(ids) for ids in pred_ids]

    max_len = max(len(x) for x in pred_ids)
    pad_id = tokenizer.pad_token_id
    pred = np.array([x + [pad_id] * (max_len - len(x)) for x in pred_ids])
    labels = np.array([x + [-100] * (max_len - len(x)) for x in label_ids])

    scores = metrics((pred, labels))
    assert scores["rouge1"] >= 0.99, scores
    assert scores["rougeL"] >= 0.99, scores


def test_metrics_handles_padding_and_oob_ids():
    pad_id = tokenizer.pad_token_id
    vocab = tokenizer.vocab_size
    pred = np.array([
        [tokenizer.bos_token_id, 5, 6, 7, vocab + 100, -1, pad_id],
    ])
    labels = np.array([
        [tokenizer.bos_token_id, 5, 6, 7, -100, -100, -100],
    ])
    scores = metrics((pred, labels))
    for k in ("rouge1", "rouge2", "rougeL", "rougeLsum"):
        assert 0.0 <= scores[k] <= 1.0, scores


def test_end_to_end_rouge_on_small_batch_beats_baseline():
    n = 10
    preds, refs = [], []
    for i in range(n):
        ex = dataset["validation"][i]
        preds.append(_summarise(ex["article"]))
        refs.append(ex["summary"])

    pred_ids = tokenizer(
        preds,
        max_length=max_target_length,
        padding=True,
        truncation=True,
        return_tensors="np",
    )["input_ids"]
    label_ids = tokenizer(
        refs,
        max_length=max_target_length,
        padding=True,
        truncation=True,
        return_tensors="np",
    )["input_ids"]
    label_ids = np.where(
        label_ids == tokenizer.pad_token_id, -100, label_ids
    )
    scores = metrics((pred_ids, label_ids))
    assert scores["rouge1"] > 0.20, scores
    assert scores["rouge2"] > 0.05, scores


# ---------- runner ----------

def _run_tests(tests):
    passed, failed = 0, []
    for name, fn in tests:
        t0 = time.time()
        try:
            fn()
            dt = time.time() - t0
            print(f"  PASS  {name}  ({dt:.2f}s)")
            passed += 1
        except AssertionError as e:
            print(f"  FAIL  {name}: {e}")
            failed.append(name)
        except Exception as e:
            print(f"  ERROR {name}: {type(e).__name__}: {e}")
            failed.append(name)
    print(f"\n{passed}/{len(tests)} passed")
    if failed:
        print("Failed:", failed)
    return not failed


tests = [
    ("tokenizer_roundtrip", test_tokenizer_roundtrip),
    ("model_forward_shapes_and_finite_loss",
     test_model_forward_shapes_and_finite_loss),
    ("generation_non_empty_and_within_length",
     test_generation_non_empty_and_within_length),
    ("generation_compresses_article",
     test_generation_compresses_article),
    ("different_articles_yield_different_summaries",
     test_different_articles_yield_different_summaries),
    ("generation_is_deterministic_with_beam_search",
     test_generation_is_deterministic_with_beam_search),
    ("metrics_perfect_score_on_identical_inputs",
     test_metrics_perfect_score_on_identical_inputs),
    ("metrics_handles_padding_and_oob_ids",
     test_metrics_handles_padding_and_oob_ids),
    ("end_to_end_rouge_on_small_batch_beats_baseline",
     test_end_to_end_rouge_on_small_batch_beats_baseline),
]

_run_tests(tests)